In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy
import pandas
from scipy.stats import gaussian_kde, ks_2samp, mannwhitneyu
import xarray as xr

In [2]:
# Define directories
Diri = '../ExtraTrack_Data/Output_Files_V7/'
Compo_Diri = '/glade/campaign/univ/upsu0032/Hyperion_ET/composites/'
Output_Diri = '../RCP_Figs/Analysis_Figs_V7.7.2/'

In [3]:
# Open file
def Open_File(File):
    DF = pandas.read_csv(File)
    DF = DF.drop("Unnamed: 0", axis=1)
    return (DF)

In [4]:
# Open each file
def Files_Open(Scenario, Diri, Subset):
    Data_DF = Open_File(Diri+Scenario+'_Data_'+Subset+'_Output.csv')
    ET_DF = Open_File(Diri+Scenario+'_ET_'+Subset+'_Output.csv')
    Codes_DF = Open_File(Diri+Scenario+'_Codes_Output.csv')
# Edit time format
    Time_Cols = ["ET Begin Time", "ET Complete Time", "Trop Peak Time", "Peak Time", "Genesis Time", "Final Time"]
    for Col in Time_Cols:
        ET_DF[Col] = pandas.to_datetime(ET_DF[Col], errors="coerce")
    Data_DF["Time(Z)"] = pandas.to_datetime(Data_DF["Time(Z)"], errors="coerce")
    return (Data_DF, ET_DF, Codes_DF)

In [5]:
# Create bins
def Create_Bins(Min, Max, Bin_Width):
    Bins = numpy.arange(Min, Max+Bin_Width, Bin_Width)
    return (Bins)

In [6]:
# Find distance between two points
def Find_Distance(y1, y2, x1, x2):
    Start_Lat = y1 * numpy.pi / 180
    End_Lat = y2 * numpy.pi / 180
    Start_Lon = x1 * numpy.pi / 180
    End_Lon = x2 * numpy.pi / 180
    Lat_Diff = End_Lat - Start_Lat
    Lon_Diff = End_Lon - Start_Lon
    Earth_Rad = 6378
    Distance = 2 * Earth_Rad * numpy.sqrt((numpy.sin(Lat_Diff/2))**2 + \
    numpy.cos(Start_Lat) * numpy.cos(End_Lat) * (numpy.sin(Lon_Diff/2))**2)
    return (Distance)

In [7]:
# Function to calculate gridbox size
def Grid_Size(Grid_Count):
    Gridbox = (0.3 * 111.32) ** 2
    Area = Grid_Count * Gridbox
    return (Area)

In [8]:
# Number of years for each climate scenario
Num_Years = numpy.array([90,93,93])

In [9]:
# Open files
Control_Data, Control_ET, Control_Codes = Files_Open("Control", Diri, "SubsetC")
RCP45_Data, RCP45_ET, RCP45_Codes = Files_Open("RCP45", Diri, "SubsetC")
RCP85_Data, RCP85_ET, RCP85_Codes = Files_Open("RCP85", Diri, "SubsetC")

In [10]:
# Create function to open composite files
def Open_Compo_File(Compo_Diri, File):
    Compo_File = xr.open_dataset(Compo_Diri + File)
    return (Compo_File)

In [11]:
# Open composite files
Control_nc_001 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.nc')
Control_nc_002 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.002.nc')
Control_nc_003 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.004.nc')

In [12]:
# Open composite files
RCP45_nc_001 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.RCP45.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.nc')
RCP45_nc_002 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.RCP45.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.002.nc')
RCP45_nc_003 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.RCP45.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.003.nc')

In [13]:
# Open composite files
RCP85_nc_001 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.RCP85.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.nc')
RCP85_nc_002 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.RCP85.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.003.nc')
RCP85_nc_003 = Open_Compo_File(Compo_Diri, 'composite_h3_CHEY.RCP85.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.004.nc')

In [14]:
Control_nc_001

<xarray.Dataset>
Dimensions:        (x: 80, y: 80, snapshot: 10562)
Coordinates:
  * x              (x) float64 -11.85 -11.55 -11.25 -10.95 ... 11.25 11.55 11.85
  * y              (y) float64 -11.85 -11.55 -11.25 -10.95 ... 11.25 11.55 11.85
Dimensions without coordinates: snapshot
Data variables: (12/62)
    snap_pathid    (snapshot) int32 ...
    snap_lon       (snapshot) float64 ...
    snap_lat       (snapshot) float64 ...
    snap_time      (snapshot) datetime64[ns] ...
    snap_U850      (snapshot, y, x) float32 ...
    snap_U500      (snapshot, y, x) float32 ...
    ...             ...
    PRECL          (y, x) float32 ...
    FLUT           (y, x) float32 ...
    CLDTOT         (y, x) float32 ...
    TMQ            (y, x) float32 ...
    OMEGA850       (y, x) float32 ...
    OMEGA500       (y, x) float32 ...

In [15]:
# Create dataframe with lat lon time data of the NetCDF files
def Composite_DF(nc, Ensemble, Year_Diff):
    Snap_Time = pandas.Series(nc.snap_time)
    Snap_Lon = pandas.Series(nc.snap_lon)
    Snap_Lat = pandas.Series(nc.snap_lat)
    Snap_PathID = pandas.Series(nc.snap_pathid)
    Index = numpy.arange(0,len(Snap_Time),1)
    Ensembles = []
    New_Time = []
    for i in range(len(Index)):
        Ensembles.append(Ensemble)
        New_Time.append(Update_Year(Snap_Time[i], Year_Diff))
    Compo_DF = pandas.DataFrame({"Index": Index, "Ensemble": Ensembles, \
    "Time": New_Time, "Lon": Snap_Lon, "Lat": Snap_Lat, "PathID": Snap_PathID})
    return (Compo_DF)

In [16]:
# Change year of data
def Update_Year(Orig_Time, Year_Diff):
    Year_Update = Orig_Time.year + Year_Diff
    New_Time = Orig_Time.replace(year=Year_Update)
    return (New_Time)

In [17]:
# Combine composite dataframes
def Combine_Compo_DF(nc_001, nc_002, nc_003, Year_Diffs):
    Compo_DF_001 = Composite_DF(nc_001, 1, Year_Diffs[0])
    Compo_DF_002 = Composite_DF(nc_002, 2, Year_Diffs[1])
    Compo_DF_003 = Composite_DF(nc_003, 3, Year_Diffs[2])
    Compo_DF = pandas.concat([Compo_DF_001, Compo_DF_002, Compo_DF_003]).reset_index(drop=True)
    return (Compo_DF)

In [18]:
Year_Diffs = [[-84,-52,-20], [-68,-36,-4], [32,64,96]]

In [19]:
# Create composite dataframes
Control_Compo = Combine_Compo_DF(Control_nc_001, Control_nc_002, Control_nc_003, Year_Diffs[0])
RCP45_Compo = Combine_Compo_DF(RCP45_nc_001, RCP45_nc_002, RCP45_nc_003, Year_Diffs[1])
RCP85_Compo = Combine_Compo_DF(RCP85_nc_001, RCP85_nc_002, RCP85_nc_003, Year_Diffs[2])

In [20]:
Control_Compo

,Index,Ensemble,Time,Lon,Lat,PathID
0,0,1,1901-08-24 18:00:00,-75.04,21.75,0
1,1,1,1901-08-25 00:00:00,-74.91,22.04,0
2,2,1,1901-08-25 06:00:00,-74.78,22.34,0
3,3,1,1901-08-25 12:00:00,-74.38,23.11,0
4,4,1,1901-08-25 18:00:00,-73.83,24.41,0
...,...,...,...,...,...,...
31218,10497,3,1994-12-15 06:00:00,-41.50,25.75,296
31219,10498,3,1994-12-15 12:00:00,-42.00,26.50,296
31220,10499,3,1994-12-15 18:00:00,-42.75,28.00,296
31221,10500,3,1994-12-16 00:00:00,-43.25,29.25,296


In [21]:
Control_ET

,Code,Name,Orig Code,Ensemble,Trans Type,Peak Time,Peak SLP,Peak Lon,Peak Lat,Trop Peak Time,...,Trop Begin Lon,Trop Begin Lat,ET Begin Time,ET Begin SLP,ET Begin Lon,ET Begin Lat,ET Complete Time,ET Complete SLP,ET Complete Lon,ET Complete Lat
0,TC190102,Beatrice,26,1,2,1901-09-03 06:00:00,933.33,-53.61,30.15,1901-09-03 06:00:00,...,-27.73,17.28,1901-09-04 06:00:00,943.55,-50.41,35.84,1901-09-10 06:00:00,978.17,-16.15,55.43
1,TC190106,Ernest,38,1,2,1901-11-22 18:00:00,991.58,-36.06,34.02,1901-11-22 18:00:00,...,-35.72,24.28,1901-11-24 18:00:00,998.89,-42.28,44.18,1901-11-25 00:00:00,998.75,-42.63,48.24
2,TC190203,Kinen,79,1,2,1902-10-13 18:00:00,974.73,-35.58,40.53,1902-10-13 18:00:00,...,-39.96,26.88,1902-10-15 06:00:00,984.17,-21.70,47.01,1902-10-16 00:00:00,999.49,-13.75,44.50
3,TC190305,Nicole,120,1,2,1903-08-17 06:00:00,903.29,-66.04,28.17,1903-08-17 06:00:00,...,-48.02,18.24,1903-08-20 00:00:00,952.77,-62.46,38.07,1903-08-21 06:00:00,968.29,-49.44,50.48
4,TC190306,Quinn,124,1,2,1903-09-05 00:00:00,951.82,-53.32,42.13,1903-09-03 18:00:00,...,-36.41,13.20,1903-09-04 12:00:00,971.30,-59.24,38.07,1903-09-05 18:00:00,972.26,-39.48,53.67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,TC199204,Daniel,1462,3,2,1992-07-12 06:00:00,986.02,-71.27,38.55,1992-07-11 12:00:00,...,-81.69,34.35,1992-07-11 18:00:00,995.10,-75.34,35.56,1992-07-13 12:00:00,1003.53,-59.03,43.68
214,TC199206,Emily,1475,3,2,1992-09-26 06:00:00,991.29,-76.18,34.46,1992-09-26 06:00:00,...,-74.09,32.66,1992-09-28 12:00:00,998.24,-75.83,37.88,1992-09-29 00:00:00,997.46,-73.94,39.36
215,TC199307,Luca,1514,3,2,1993-08-23 12:00:00,951.47,-71.26,25.86,1993-08-23 12:00:00,...,-50.92,15.44,1993-08-30 06:00:00,983.81,-53.91,48.22,1993-08-30 12:00:00,983.91,-55.38,50.20
216,TC199404,Tess,1568,3,2,1994-08-24 00:00:00,950.53,-61.46,36.80,1994-08-24 00:00:00,...,-54.01,18.37,1994-08-24 12:00:00,952.17,-58.80,38.89,1994-08-27 12:00:00,1002.63,-36.27,46.14


In [22]:
# Create function to find index of selected snapshot
def Find_Snapshot(Compo_DF, Ensemble, Time, Lon, Lat):
# Find possible storms that occur at the same time
    Compo_Storm = Compo_DF[(Compo_DF["Ensemble"] == Ensemble) & (Compo_DF["Time"] == Time)].reset_index(drop=True)
# If no storm found: no index
    if len(Compo_Storm) == 0:
        return (numpy.nan)
# Storms found: find storm closest to storm center
    else:
        Dists = []
        for c in range(len(Compo_Storm)):
            Dists.append(Find_Distance(Lat, Compo_Storm["Lat"][c], Lon, Compo_Storm["Lon"][c]))
        Min_Idx = numpy.array(Dists).argmin()
        Min_Dist = Dists[Min_Idx]
# At most 300km of error in location permitted
        if Min_Dist <= 300:
            return (int(Compo_Storm["Index"][Min_Idx]))
        else:
            return (numpy.nan)

In [23]:
# Create function to find index of tropical peak, ET initiation and ET completion snapshots
def ET_Snapshots_DF(ET_DF, Compo_DF):
    Ensembles = ET_DF["Ensemble"]
    Trop_Peak_Times = ET_DF["Trop Peak Time"]
    Trop_Peak_DF = ET_DF[["Trop Peak SLP", "Trop Peak Lon", "Trop Peak Lat"]].to_numpy().T
    ET_Begin_Times = ET_DF["ET Begin Time"]
    ET_Begin_DF = ET_DF[["ET Begin SLP", "ET Begin Lon", "ET Begin Lat"]].to_numpy().T
    ET_Compl_Times = ET_DF["ET Complete Time"]
    ET_Compl_DF = ET_DF[["ET Complete SLP", "ET Complete Lon", "ET Complete Lat"]].to_numpy().T
#
# Create empty array to store indices
    Index = numpy.zeros((3, len(ET_DF)))
    for i in range(len(ET_DF)):
# Find index of tropical peak points
        Index[0][i] = Find_Snapshot(Compo_DF, Ensembles[i], Trop_Peak_Times[i], Trop_Peak_DF[1][i], Trop_Peak_DF[2][i])
# Find index of ET initiation points
        Index[1][i] = Find_Snapshot(Compo_DF, Ensembles[i], ET_Begin_Times[i], ET_Begin_DF[1][i], ET_Begin_DF[2][i])
# Find index of ET completion points
        Index[2][i] = Find_Snapshot(Compo_DF, Ensembles[i], ET_Compl_Times[i], ET_Compl_DF[1][i], ET_Compl_DF[2][i])
#
# Create new dataframes for tropical peak, ET initiation and ET completion to store output
    Cols = ["Code", "Name", "Orig Code", "Ensemble"]
    Trop_Peak_Cols = ["Trop Peak Time", "Trop Peak SLP", "Trop Peak Lon", "Trop Peak Lat"]
    ET_Begin_Cols = ["ET Begin Time", "ET Begin SLP", "ET Begin Lon", "ET Begin Lat"]
    ET_Compl_Cols = ["ET Complete Time", "ET Complete SLP", "ET Complete Lon", "ET Complete Lat"]
    Trop_Peak_DF = ET_DF[Cols + Trop_Peak_Cols].copy()
    ET_Begin_DF = ET_DF[Cols + ET_Begin_Cols].copy()
    ET_Compl_DF = ET_DF[Cols + ET_Compl_Cols].copy()
    Trop_Peak_DF["Index"] = Index[0]
    Trop_Peak_DF["Index"] = Trop_Peak_DF["Index"].astype("Int64")
    ET_Begin_DF["Index"] = Index[1]
    ET_Begin_DF["Index"] = ET_Begin_DF["Index"].astype("Int64")
    ET_Compl_DF["Index"] = Index[2]
    ET_Compl_DF["Index"] = ET_Compl_DF["Index"].astype("Int64")
    return (Trop_Peak_DF, ET_Begin_DF, ET_Compl_DF)

In [24]:
# Create dataframes with indices of snapshots
Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl = ET_Snapshots_DF(Control_ET, Control_Compo)
RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl = ET_Snapshots_DF(RCP45_ET, RCP45_Compo)
RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl = ET_Snapshots_DF(RCP85_ET, RCP85_Compo)

In [25]:
Control_Trop_Peak

,Code,Name,Orig Code,Ensemble,Trop Peak Time,Trop Peak SLP,Trop Peak Lon,Trop Peak Lat,Index
0,TC190102,Beatrice,26,1,1901-09-03 06:00:00,933.33,-53.61,30.15,74
1,TC190106,Ernest,38,1,1901-11-22 18:00:00,991.58,-36.06,34.02,178
2,TC190203,Kinen,79,1,1902-10-13 18:00:00,974.73,-35.58,40.53,353
3,TC190305,Nicole,120,1,1903-08-17 06:00:00,903.29,-66.04,28.17,543
4,TC190306,Quinn,124,1,1903-09-03 18:00:00,957.33,-65.07,34.07,619
...,...,...,...,...,...,...,...,...,...
213,TC199204,Daniel,1462,3,1992-07-11 12:00:00,998.19,-77.70,34.63,9633
214,TC199206,Emily,1475,3,1992-09-26 06:00:00,991.29,-76.18,34.46,9701
215,TC199307,Luca,1514,3,1993-08-23 12:00:00,951.47,-71.26,25.86,9927
216,TC199404,Tess,1568,3,1994-08-24 00:00:00,950.53,-61.46,36.80,10235


In [26]:
# Calculate 850hPa windspeed
def Windspeed_850hPa(nc, Index):
    U850 = numpy.array(nc.snap_U850[Index])
    V850 = numpy.array(nc.snap_V850[Index])
    Wind850 = numpy.sqrt(U850 **2 + V850 **2)
    return (Wind850)

In [27]:
# Find 850hPa max windspeed and wind field sizes
def Wind_Field_Find(Trop_Peak_DF, ET_Begin_DF, ET_Compl_DF, nc_001, nc_002, nc_003):
    Wind_Field_Info = numpy.zeros((3,len(Trop_Peak_DF),5))
    nc_Map = {1: nc_001, 2: nc_002, 3: nc_003}
    DF_Map = {0: Trop_Peak_DF, 1: ET_Begin_DF, 2: ET_Compl_DF}
#
# Loop over each storm
    for i in range(len(Trop_Peak_DF)):
# Loop over tropical peak, ET initiation and ET completion
        for j in range(3):
# Get index
            DF = DF_Map[j]
            Index = DF["Index"][i]
            nc = nc_Map[DF["Ensemble"][i]]
# Find 850hPa windspeed snapshot
            Snap_850 = Windspeed_850hPa(nc, Index)
# Find maximum 850hPa windspeed
            Max_Wind_850 = numpy.max(Snap_850)
            Wind_Field_Info[j][i][0] = Max_Wind_850
# Count number of datapoints with windspeed above 13,18,25,33m/s
            Snap_Sort = numpy.sort(Snap_850.ravel())
            Count_13 = len(Snap_Sort[Snap_Sort >= 13])
            Count_18 = len(Snap_Sort[Snap_Sort >= 18])
            Count_25 = len(Snap_Sort[Snap_Sort >= 25])
            Count_33 = len(Snap_Sort[Snap_Sort >= 33])
            Wind_Field_Info[j][i][1] = Grid_Size(Count_13) * 10**-4
            Wind_Field_Info[j][i][2] = Grid_Size(Count_18) * 10**-4
            Wind_Field_Info[j][i][3] = Grid_Size(Count_25) * 10**-4
            Wind_Field_Info[j][i][4] = Grid_Size(Count_33) * 10**-4
#
# Add 850hPa wind field info into dataframes
        Wind_Field_Cols = ["Max 850hPa Winds", "13m/s Wind Field", "18m/s Wind Field", "25m/s Wind Field", "33m/s Wind Field"]
        Trop_Peak_DF[Wind_Field_Cols] = Wind_Field_Info[0]
        ET_Begin_DF[Wind_Field_Cols] = Wind_Field_Info[1]
        ET_Compl_DF[Wind_Field_Cols] = Wind_Field_Info[2]
    return (Trop_Peak_DF, ET_Begin_DF, ET_Compl_DF)

In [28]:
# Calculate precipitation rate
def Precip_Rate(nc, Index):
    Precip_ms = numpy.array(nc.snap_PRECT[int(Index)])
    Precip_mmhr = Precip_ms * 3600 * 1000
    return (Precip_mmhr)

In [29]:
# Find maximum precip rate and precip field sizes
def Precip_Field_Find(Trop_Peak_DF, ET_Begin_DF, ET_Compl_DF, nc_001, nc_002, nc_003):
    Precip_Field_Info = numpy.zeros((3,len(Trop_Peak_DF),5))
    nc_Map = {1: nc_001, 2: nc_002, 3: nc_003}
    DF_Map = {0: Trop_Peak_DF, 1: ET_Begin_DF, 2: ET_Compl_DF}
#
# Loop over each storm
    for i in range(len(Trop_Peak_DF)):
# Loop over tropical peak, ET initiation and ET completion
        for j in range(3):
# Get index
            DF = DF_Map[j]
            Index = DF["Index"][i]
            nc = nc_Map[DF["Ensemble"][i]]
# Find precip rate snapshot
            Snap_Precip = Precip_Rate(nc, Index)
# Find maximum precip rate
            Max_Precip = numpy.max(Snap_Precip)
            Precip_Field_Info[j][i][0] = Max_Precip
# Count number of datapoints with precip rate above 0.5,1,5,10mm/hr
            Snap_Sort = numpy.sort(Snap_Precip.ravel())
            Count_Half = len(Snap_Sort[Snap_Sort >= 0.5])
            Count_1 = len(Snap_Sort[Snap_Sort >= 1])
            Count_5 = len(Snap_Sort[Snap_Sort >= 5])
            Count_10 = len(Snap_Sort[Snap_Sort >= 10])
            Precip_Field_Info[j][i][1] = Grid_Size(Count_Half) * 10**-4
            Precip_Field_Info[j][i][2] = Grid_Size(Count_1) * 10**-4
            Precip_Field_Info[j][i][3] = Grid_Size(Count_5) * 10**-4
            Precip_Field_Info[j][i][4] = Grid_Size(Count_10) * 10**-4
#
# Add precip field info into dataframes
        Precip_Field_Cols = ["Max Precip Rate", "0.5mm/hr Precip Field", "1mm/hr Precip Field", "5mm/hr Precip Field", "10mm/hr Precip Field"]
        Trop_Peak_DF[Precip_Field_Cols] = Precip_Field_Info[0]
        ET_Begin_DF[Precip_Field_Cols] = Precip_Field_Info[1]
        ET_Compl_DF[Precip_Field_Cols] = Precip_Field_Info[2]
    return (Trop_Peak_DF, ET_Begin_DF, ET_Compl_DF)

In [ ]:
# Add 850hPa wind field info into dataframes
Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl = \
Wind_Field_Find(Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl, Control_nc_001, Control_nc_002, Control_nc_003)
RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl = \
Wind_Field_Find(RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl, RCP45_nc_001, RCP45_nc_002, RCP45_nc_003)
RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl = \
Wind_Field_Find(RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl, RCP85_nc_001, RCP85_nc_002, RCP85_nc_003)

In [ ]:
# Add precip field info into dataframes
Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl = \
Precip_Field_Find(Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl, Control_nc_001, Control_nc_002, Control_nc_003)
RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl = \
Precip_Field_Find(RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl, RCP45_nc_001, RCP45_nc_002, RCP45_nc_003)
RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl = \
Precip_Field_Find(RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl, RCP85_nc_001, RCP85_nc_002, RCP85_nc_003)

In [ ]:
Control_ET_Compl

In [ ]:
# Function for computing smooth probability distribution and approximating smooth cumulative distribution
def KDE_PDF_CDF(Data, Min, Max, Bin_Width, Smooth_Param):
# Compute PDF
    Values = Create_Bins(Min, Max, Bin_Width)
    KDE = gaussian_kde(Data, bw_method=Smooth_Param)
    PDF = KDE(Values)
# Numerical integration to obtain cumulative distribution
    dx = numpy.diff(Values)
    CDF = numpy.zeros(len(Values))
    CDF[1:] = numpy.cumsum(0.5 * (PDF[:-1] + PDF[1:]) * dx)
    CDF = CDF / CDF[-1]
    return (Values, PDF, CDF)

In [ ]:
# Function for plotting cumulative distribution frequency
def Plot_CDF(Axis, Control_Array, RCP45_Array, RCP85_Array, Min, Max, Bin_Width, Smooth_Param):
    Control_Values, Control_PDF, Control_CDF = KDE_PDF_CDF(numpy.array(Control_Array), Min, Max, Bin_Width, Smooth_Param)
    RCP45_Values, RCP45_PDF, RCP45_CDF = KDE_PDF_CDF(numpy.array(RCP45_Array), Min, Max, Bin_Width, Smooth_Param)
    RCP85_Values, RCP85_PDF, RCP85_CDF = KDE_PDF_CDF(numpy.array(RCP85_Array), Min, Max, Bin_Width, Smooth_Param)
    Axis.plot(Control_Values, Control_CDF, color='limegreen', linewidth=2.8, label='Control', alpha=0.8)
    Axis.plot(RCP45_Values, RCP45_CDF, color='orange', linewidth=2.8, label='RCP4.5', alpha=0.8)
    Axis.plot(RCP85_Values, RCP85_CDF, color='red', linewidth=2.8, label='RCP8.5', alpha=0.8)

In [ ]:
# Create function to plot cumulative and probability distributions of snapshots
def Snap_Distr_Plot(Control_DFs, RCP45_DFs, RCP85_DFs, Vars, Snaps, \
    Mins, Maxes, Bin_Widths, Label_Widths, x_Labels, Smooth_Param, Savefig, Figname):
    Fig = pyplot.figure(figsize=(16,16))
    Axes = Fig.subplots(3,3, sharey=True)
#
# Find dataframe and variable
    for i in range(3):
        for j in range(3):
            Control_DF = Control_DFs[i]
            RCP45_DF = RCP45_DFs[i]
            RCP85_DF = RCP85_DFs[i]
            Snap = Snaps[i]
            Var = Vars[j]
# Plotting
            Plot_CDF(Axes[j][i], Control_DF[Var], RCP45_DF[Var], RCP85_DF[Var], Mins[j], Maxes[j], Bin_Widths[j], Smooth_Param)
            CDF_Formatting(Axes[j][i], i, j, Var, Mins[j], Maxes[j], Label_Widths[j], x_Labels[j], str(Snap + " " + Var))
#
# Save fig
    Fig.tight_layout()
    if Savefig == True:
        Fig.savefig(Output_Diri+Figname, bbox_inches='tight')

In [ ]:
# Formatting based on variable
def CDF_Formatting(Axis, i, j, Var, Min, Max, Width, x_Label, Title):
# ticks and limits
    x_Ticks = Create_Bins(Min, Max, Width)
    Axis.set_xticks(x_Ticks)
    Axis.set_xlim(Min, Max)
    Axis.set_yticks(Create_Bins(0,1,0.125))
    Axis.set_ylim(0,1)
#
# Labels and title
    Axis.set_xlabel(x_Label, fontsize=12)
    if i == 0:
        Axis.set_ylabel("Cumulative Frequency Density", fontsize=12)
    Axis.set_title(Title, fontsize=18)
#
# Gridlines
    Axis.grid(linewidth=0.3, color='silver', linestyle='-')
#
# Letter labels
    Letter_Label(Axis, i+j*3)
#
# Legend
    if i == 0 and j == 0:
        Axis.legend(loc=4, fontsize=15)

In [ ]:
# Letter labels
def Letter_Label(Axis, Plot):
    Fig_Labels = ['(a)','(b)','(c)','(d)', '(e)', '(f)', '(g)', '(h)', '(i)', '(j)', '(k)', '(l)']
    Axis.text(0.05, 0.95, str(Fig_Labels[int(Plot)]), ha='center', va='center', \
    transform=Axis.transAxes, fontdict={'size':24},color='black')

In [ ]:
Control_DFs = [Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl]
RCP45_DFs = [RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl]
RCP85_DFs = [RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl]
Vars = ["Max 850hPa Winds", "18m/s Wind Field", "33m/s Wind Field"]
Snaps = ["Tropical Peak", "ET Initiation", "ET Completion"]
Mins = [0,0,0]
Maxes = [100,600,120]
Bin_Widths = [1,5,1]
Label_Widths = [10,50,10]
x_Labels = ["Windspeed ($m/s$)", "Wind Field Size ($10^{4} km^{2}$)", "Wind Field Size ($10^{4} km^{2}$)"]
Smooth_Param = 0.10
Savefig = True
Figname = "Wind_850_CDF.png"
Snap_Distr_Plot(Control_DFs, RCP45_DFs, RCP85_DFs, Vars, Snaps, \
Mins, Maxes, Bin_Widths, Label_Widths, x_Labels, Smooth_Param, Savefig, Figname)

In [ ]:
Control_DFs = [Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl]
RCP45_DFs = [RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl]
RCP85_DFs = [RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl]
Vars = ["Max Precip Rate", "1mm/hr Precip Field", "10mm/hr Precip Field"]
Snaps = ["Tropical Peak", "ET Initiation", "ET Completion"]
Mins = [0,0,0]
Maxes = [100,160,16]
Bin_Widths = [1,2,0.2]
Label_Widths = [10,20,2]
x_Labels = ["Precip Rate ($mm/hr$)", "Precip Field Size ($10^{4} km^{2}$)", "Precip Field Size ($10^{4} km^{2}$)"]
Smooth_Param = 0.10
Savefig = True
Figname = "Precip_Rate_CDF.png"
Snap_Distr_Plot(Control_DFs, RCP45_DFs, RCP85_DFs, Vars, Snaps, \
Mins, Maxes, Bin_Widths, Label_Widths, x_Labels, Smooth_Param, Savefig, Figname)

In [ ]:
# Bootstrap test for statistical significance between medians
def Bootstrap_Test(A, B, n=10000, Random_Seed=28):
    Actual_Diff = numpy.median(B) - numpy.median(A)
    Boot_Diffs = numpy.zeros(n)
    rng = numpy.random.default_rng(Random_Seed)
# Bootstrap test
    for i in range(n):
        A_Sample = rng.choice(A, size=len(A), replace=True)
        B_Sample = rng.choice(B, size=len(B), replace=True)
        Boot_Diffs[i] = numpy.median(B_Sample) - numpy.median(A_Sample)
# Get confidence interval
    CI_Low, CI_High = numpy.percentile(Boot_Diffs[i], [2.5, 97.5])
# Approximate two sided P value
    if Actual_Diff >= 0:
        P_Val = 2 * numpy.mean(Boot_Diffs <= 0)
    else:
        P_Val = 2 * numpy.mean(Boot_Diffs >= 0)
    P_Val = numpy.clip(P_Val, 0, 1)
    return (P_Val)

In [ ]:
# Find statistical significance for ET cumulative distributions
def ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, Var):
# Create empty arrays
    Medians = numpy.zeros((3,3))
    KS_P_Vals = numpy.zeros((2,3))
    MW_P_Vals = numpy.zeros((2,3))
    Boot_P_Vals = numpy.zeros((2,3))
#
# Find statistical significance for genesis, peak, tropical peak, ET initiation and ET completion
    Snaps = ["Trop Peak", "ET Begin", "ET Complete"]
    SLP_Bounds = [1000, 1010, 1010]
    for i in range(3):
        Control_DF = Control_DFs[i]
        RCP45_DF = RCP45_DFs[i]
        RCP85_DF = RCP85_DFs[i]
# Compute medians
        Medians[0][i] = numpy.median(Control_DF[Var])
        Medians[1][i] = numpy.median(RCP45_DF[Var])
        Medians[2][i] = numpy.median(RCP85_DF[Var])
# Compute statistical significance with KS Test
        KS_P_Vals[0][i] = ks_2samp(Control_DF[Var], RCP45_DF[Var])[1]
        KS_P_Vals[1][i] = ks_2samp(Control_DF[Var], RCP85_DF[Var])[1]
# Compute statistical significance with Mann Whitney Test
        MW_P_Vals[0][i] = mannwhitneyu(Control_DF[Var], RCP45_DF[Var])[1]
        MW_P_Vals[1][i] = mannwhitneyu(Control_DF[Var], RCP85_DF[Var])[1]
# Compute statistical significance with bootstrap test
        Boot_P_Vals[0][i] = Bootstrap_Test(Control_DF[Var], RCP45_DF[Var])
        Boot_P_Vals[1][i] = Bootstrap_Test(Control_DF[Var], RCP85_DF[Var])
#
# Output as dataframe
    Signif_DF = pandas.DataFrame({"Varable": [Var, Var, Var], "Snapshot": Snaps, \
                "Median (Control)": Medians[0], "Median (RCP4.5)": Medians[1], "Median (RCP8.5)": Medians[2], \
                "KS Test (RCP4.5 - Control)": KS_P_Vals[0], "KS Test (RCP8.5 - Control)": KS_P_Vals[1], \
                #"MW Test (RCP4.5 - Control)": MW_P_Vals[0], "MW Test (RCP8.5 - Control)": MW_P_Vals[1], \
                "Bootstrap Test (RCP4.5 - Control)": Boot_P_Vals[0], "Bootstrap Test (RCP8.5 - Control)": Boot_P_Vals[1]})
    Signif_DF = Signif_DF.round(3)
    return (Signif_DF)

In [ ]:
Control_DFs = [Control_Trop_Peak, Control_ET_Begin, Control_ET_Compl]
RCP45_DFs = [RCP45_Trop_Peak, RCP45_ET_Begin, RCP45_ET_Compl]
RCP85_DFs = [RCP85_Trop_Peak, RCP85_ET_Begin, RCP85_ET_Compl]

In [ ]:
Wind_Max_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "Max 850hPa Winds")
Wind_Max_Signif

In [ ]:
Wind_13_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "13m/s Wind Field")
Wind_13_Signif

In [ ]:
Wind_18_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "18m/s Wind Field")
Wind_18_Signif

In [ ]:
Wind_25_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "25m/s Wind Field")
Wind_25_Signif

In [ ]:
Wind_33_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "33m/s Wind Field")
Wind_33_Signif

In [ ]:
Precip_Max_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "Max Precip Rate")
Precip_Max_Signif

In [ ]:
Precip_Half_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "0.5mm/hr Precip Field")
Precip_Half_Signif

In [ ]:
Precip_1_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "1mm/hr Precip Field")
Precip_1_Signif

In [ ]:
Precip_5_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "5mm/hr Precip Field")
Precip_5_Signif

In [ ]:
Precip_10_Signif = ET_Diff_Signif(Control_DFs, RCP45_DFs, RCP85_DFs, "10mm/hr Precip Field")
Precip_10_Signif

In [ ]:
# Create output dataframe
Signif_DF = pandas.concat([Wind_Max_Signif, Wind_18_Signif, Wind_33_Signif, Precip_Max_Signif, Precip_1_Signif, Precip_10_Signif])
Signif_DF

In [ ]:
# Output DF to csv file
def Output_File(DF, File_Name):
    DF.to_csv(Output_Diri+File_Name)

In [ ]:
Output_File(Signif_DF, 'Composite_Signif.csv')